# Notebook 2 — PlanetScope training and evaluation

This notebook trains the Planet-only damage classifier on the eight channels created by notebook 1: matching pre/post B, G, R, and NIR patches at 3 m resolution. It contains no Sentinel inputs.

The default `base_cnn` is the same control architecture as the S1 pipeline: each block is `Conv2d(3×3) → BatchNorm → ReLU → MaxPool(2)`, followed by global average pooling, dropout, and one output logit. `siamese_cnn` remains selectable and uses the same shared-encoder logic as S1.

## Spatial protocol — exactly the S1 notebook 2 bands

| role | latitude-quantile band | fitted component |
|---|---:|---|
| train | 0.55–1.00 | CNN weights |
| stack | 0.42–0.55 | XGBoost spatial stage |
| validation | 0.33–0.42 | early stopping, HPO, variant selection, threshold |
| test | 0.00–0.33 | opened once after everything is frozen |

There are no buffer gaps. Planet has only one imagery/assessment row per footprint, whereas S1 used the same bands across several assessment dates. Spatial XGBoost features are calculated separately inside each role, so neighbors never cross a split boundary.


## 1. Colab setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q optuna xgboost pyarrow geopandas rasterio scikit-learn


In [ ]:
import os
import sys
import json
import time
import hashlib
import importlib
import copy
import pickle
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)

PROJECT_DIR = '/content/drive/MyDrive/War-Damage-Detection/Planet Data (Nils)'
os.environ['PLANET_DAMAGE_BASE'] = PROJECT_DIR
os.environ['PLANET_EXPERIMENTS_DIR'] = os.path.join(PROJECT_DIR, 'experiments')
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import pipeline
importlib.reload(pipeline)
from pipeline import (
    PLANET_BANDS, PlanetPairDataset, PLANET_MODEL_REGISTRY,
    inspect_dataset, load_dataset, footprint_latitude_reference, set_random_seed,
    latitude_quantile_spatial_split, spatial_split_metadata,
    estimate_shared_percentiles, build_planet_model,
    predict_planet_probs, neighbour_features,
    best_f1_threshold, binary_metrics, experiment_dirs,
    atomic_json_dump, atomic_torch_save,
)


### 1a. One experiment configuration


In [ ]:
EVALUATE_ONLY = False

CONFIG = {
    'experiment_name': 'planet_exp001_base_cnn_spatial',
    'seed': 0,
    'model': 'base_cnn',             # or 'siamese_cnn'
    'bands': ['B', 'G', 'R', 'NIR'],

    # Exact S1 BaseCNN defaults. For siamese_cnn use, for example:
    # {'width': 32, 'depth': 3, 'dropout': .4,
    #  'head_dim': 128, 'head_dropout': .2}
    'model_params': {'width': 8, 'depth': 2, 'dropout': 0.2},

    # Do not alter these when making a directly comparable S1/Planet run.
    'split': {
        'train': [0.55, 1.00],
        'stack': [0.42, 0.55],
        'val': [0.33, 0.42],
        'test': [0.00, 0.33],
    },
    'normalization': {
        'lower_percentile': 2.0,
        'upper_percentile': 98.0,
        'sample_patches': 20_000,
    },
    'augmentation': {
        'geometric': True,
        'brightness_jitter': 0.0,
    },
    'training': {
        'batch_size': 128,
        'learning_rate': 1e-3,
        'weight_decay': 1e-4,
        'max_epochs': 120,
        'patience': 10,
        'num_workers': 2,
        'predict_batch_size': 1024,
    },
    'stacking': {
        'enabled': True,
        'ks': [8, 32],
        'n_estimators': 600,
        'max_depth': 4,
        'learning_rate': 0.05,
        'early_stopping_rounds': 40,
    },
    'hpo': {
        'enabled': False,
        'n_trials': 10,          # target total; increase to extend/resume
        'max_epochs': 25,
        'patience': 6,
        'train_subsample': 0.5,
        'timeout_minutes': None, # resumable wall-clock budget
    },
}

if CONFIG['model'] not in PLANET_MODEL_REGISTRY:
    raise ValueError(f"model must be one of {sorted(PLANET_MODEL_REGISTRY)}")
def experiment_identity(cfg):
    """Configuration fields that define fitted models and selection.

    HPO trial count and timeout are budgets, not scientific settings. They
    may increase in place so an existing Optuna study can continue.
    """
    identity = copy.deepcopy(cfg)
    identity.get('hpo', {}).pop('n_trials', None)
    identity.get('hpo', {}).pop('timeout_minutes', None)
    return identity


set_random_seed(CONFIG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = experiment_dirs(CONFIG['experiment_name'])
CONFIG_PATH = OUT['root'] / 'config.json'
FINAL_SELECTION_PATH = OUT['metrics'] / 'final_selection.json'
FINAL_MARKER_PATH = OUT['metrics'] / 'final_test_complete.json'

if FINAL_MARKER_PATH.exists() and not EVALUATE_ONLY:
    raise RuntimeError(
        'This experiment has already opened its final test. Use '
        'EVALUATE_ONLY=True to inspect saved outputs or choose a new '
        'experiment_name for further development.')

if CONFIG_PATH.exists():
    with CONFIG_PATH.open(encoding='utf-8') as handle:
        previous_config = json.load(handle)
    if experiment_identity(previous_config) != experiment_identity(CONFIG):
        raise RuntimeError(
            f'{OUT["root"]} contains a different scientific configuration. '
            'Only hpo.n_trials and hpo.timeout_minutes may change in place; '
            'otherwise choose a new experiment_name.')

    old_trials = int(previous_config.get('hpo', {}).get('n_trials', 0))
    new_trials = int(CONFIG.get('hpo', {}).get('n_trials', 0))
    if new_trials < old_trials:
        raise RuntimeError(
            f'Cannot reduce hpo.n_trials from {old_trials} to {new_trials} '
            'inside an existing experiment.')
    if previous_config != CONFIG:
        atomic_json_dump(CONFIG, CONFIG_PATH)
        print(
            f'Updated resumable HPO budget: n_trials '
            f'{old_trials} -> {new_trials}. Existing trials are retained.')
    else:
        print(
            'Compatible experiment found. Completed CNN runs will be '
            'reused and interrupted full-data training will resume.')
else:
    atomic_json_dump(CONFIG, CONFIG_PATH)
    print('Created new experiment configuration.')

print('device:', device)
print('experiment:', OUT['root'])
print('available models:', sorted(PLANET_MODEL_REGISTRY))
print('mode:', 'evaluate saved experiment' if EVALUATE_ONLY else 'train / resume')


## 2. Load notebook 1 output and fit the S1 split


In [ ]:
dataset_info = inspect_dataset(validate_alignment=True)
data = load_dataset(load_images=True)
X = data['X']
y = data['y']
table = data['gdf'].reset_index(drop=True)

if len(X) != len(y) or len(y) != len(table):
    raise RuntimeError('NPZ/parquet row alignment changed after validation')
if not np.array_equal(data['system_index'], table['system:index'].astype(str).to_numpy()):
    raise RuntimeError('NPZ and parquet footprint IDs are not row-aligned')

split_bands = {name: tuple(values) for name, values in CONFIG['split'].items()}
split_reference = footprint_latitude_reference()
split = latitude_quantile_spatial_split(
    table, bands=split_bands, reference=split_reference)
split_meta = spatial_split_metadata(split)
atomic_json_dump(split_meta, OUT['metrics'] / 'spatial_split.json')

rows = []
for role in ('train', 'stack', 'val', 'test'):
    idx = split[role]
    row = {'role': role, 'n': len(idx), 'fraction': len(idx) / len(y)}
    # Test prevalence is deliberately not inspected during development.
    row['damaged_fraction'] = float(y[idx].mean()) if role != 'test' else np.nan
    rows.append(row)
split_table = pd.DataFrame(rows)
display(split_table)
display(pd.DataFrame({'dataset': [dataset_info]}))


In [ ]:
colors = {'train': 'tab:blue', 'stack': 'tab:purple',
          'val': 'tab:orange', 'test': 'tab:red'}
fig, ax = plt.subplots(figsize=(6, 9))
for role, color in colors.items():
    idx = split[role]
    ax.scatter(table.iloc[idx]['lon'], table.iloc[idx]['lat'], s=2,
               alpha=.35, color=color, label=f'{role} ({len(idx):,})')
ax.set(xlabel='longitude', ylabel='latitude',
       title='Exact S1 latitude-quantile bands applied to Planet footprints')
ax.legend(markerscale=4)
fig.savefig(OUT['figures'] / 'spatial_split.png', dpi=180, bbox_inches='tight')
plt.show()


## 3. Training-only normalization and data loaders


In [ ]:
norm_cfg = CONFIG['normalization']
lo, hi, norm_rows = estimate_shared_percentiles(
    X, split['train'], bands=CONFIG['bands'],
    lower=norm_cfg['lower_percentile'],
    upper=norm_cfg['upper_percentile'],
    sample_patches=norm_cfg['sample_patches'], seed=CONFIG['seed'])
normalization = {
    'bands': CONFIG['bands'], 'lo': lo.tolist(), 'hi': hi.tolist(),
    'sampled_training_rows': int(len(norm_rows)),
}
atomic_json_dump(normalization, OUT['metrics'] / 'normalization.json')
display(pd.DataFrame({'band': CONFIG['bands'], 'p02': lo, 'p98': hi}))

def make_loader(indices, batch_size, augment=False, shuffle=False):
    ds = PlanetPairDataset(
        X, y, indices, bands=CONFIG['bands'], lo=lo, hi=hi,
        augment=augment,
        brightness_jitter=(CONFIG['augmentation']['brightness_jitter'] if augment else 0.0))
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle,
        num_workers=CONFIG['training']['num_workers'],
        pin_memory=(device.type == 'cuda'), persistent_workers=False)

predict_loaders = {
    role: make_loader(split[role], CONFIG['training']['predict_batch_size'])
    for role in ('stack', 'val')
}


## 4. CNN training


In [ ]:
def stratified_subsample(indices, fraction, seed):
    indices = np.asarray(indices, dtype=np.int64)
    if fraction >= 1:
        return indices
    rng = np.random.default_rng(seed)
    pieces = []
    for label in (0, 1):
        candidates = indices[y[indices] == label]
        count = max(1, int(round(fraction * len(candidates))))
        pieces.append(rng.choice(candidates, count, replace=False))
    return np.sort(np.concatenate(pieces))


def load_cnn(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model = build_planet_model(
        checkpoint['model_name'], n_channels=2 * len(checkpoint['bands']),
        **checkpoint['model_params']).to(device)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    return model, checkpoint


def cpu_state_dict(model):
    return {
        name: value.detach().cpu().clone()
        for name, value in model.state_dict().items()
    }


def train_cnn(run_name, model_params, training_params, max_epochs=None,
              patience=None, trial=None, train_fraction=1.0,
              save=True, verbose=True, log_every=None):
    """Train, reuse, or resume one full-data CNN run.

    Completed runs are immutable and reused when their signature matches.
    Full-data baseline/HPO-winner runs save current model, optimizer, AMP,
    early-stopping, history, and RNG state after every epoch. Short Optuna
    trials use SQLite persistence and deliberately do not write CNN resume
    checkpoints.
    """
    trial_number = 0 if trial is None else trial.number + 1
    set_random_seed(CONFIG['seed'] + trial_number)
    train_idx = stratified_subsample(
        split['train'], train_fraction, CONFIG['seed'])
    batch_size = int(training_params['batch_size'])
    max_epochs = int(max_epochs or CONFIG['training']['max_epochs'])
    patience = int(patience or CONFIG['training']['patience'])
    model_params = dict(model_params)
    training_params = dict(training_params)

    signature = {
        'dataset_fingerprint': data['metadata']['preprocessing_fingerprint'],
        'model_name': CONFIG['model'],
        'model_params': model_params,
        'training_params': training_params,
        'bands': list(CONFIG['bands']),
        'augmentation': copy.deepcopy(CONFIG['augmentation']),
        'normalization': normalization,
        'split': split_meta,
        'train_rows': int(len(train_idx)),
        'train_indices_sha256': hashlib.sha256(
            np.asarray(train_idx, dtype='<i8').tobytes()).hexdigest(),
        'max_epochs': max_epochs,
        'patience': patience,
        'seed': int(CONFIG['seed'] + trial_number),
    }
    final_path = OUT['models'] / f'{run_name}.pt' if save else None
    resume_path = OUT['models'] / f'{run_name}_training.pt' if save else None

    if final_path is not None and final_path.exists():
        state = torch.load(final_path, map_location='cpu', weights_only=False)
        if state.get('complete') and state.get('signature') == signature:
            if verbose:
                print(
                    f"reused completed {run_name}: epoch "
                    f"{state['best_epoch']}, val_AP={state['best_val_PR_AUC']:.4f}")
            return {
                'name': run_name, 'checkpoint': str(final_path),
                'model_params': model_params,
                'training_params': training_params,
                'best_val_PR_AUC': float(state['best_val_PR_AUC']),
                'resumed': False, 'reused': True,
            }
        if verbose:
            print(f'completed checkpoint for {run_name} is incompatible; retraining')

    train_loader = make_loader(
        train_idx, batch_size,
        augment=CONFIG['augmentation']['geometric'], shuffle=True)
    val_loader = predict_loaders['val']
    model = build_planet_model(
        CONFIG['model'], n_channels=2 * len(CONFIG['bands']),
        **model_params).to(device)

    positives = int(y[train_idx].sum())
    negatives = len(train_idx) - positives
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(negatives / max(positives, 1), device=device))
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=training_params['learning_rate'],
        weight_decay=training_params['weight_decay'])
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    start_epoch = 1
    best_ap, best_epoch, best_state = -np.inf, -1, None
    stale, history, resumed = 0, [], False
    if resume_path is not None and resume_path.exists():
        state = torch.load(resume_path, map_location='cpu', weights_only=False)
        if state.get('signature') == signature and not state.get('complete', False):
            model.load_state_dict(state['model_state'])
            optimizer.load_state_dict(state['optimizer_state'])
            if state.get('scaler_state'):
                scaler.load_state_dict(state['scaler_state'])
            start_epoch = int(state['completed_epoch']) + 1
            best_ap = float(state['best_val_PR_AUC'])
            best_epoch = int(state['best_epoch'])
            best_state = state['best_model_state']
            stale = int(state['stale_epochs'])
            history = list(state['history'])
            torch.set_rng_state(state['torch_rng_state'])
            if use_amp and state.get('cuda_rng_state_all') is not None:
                torch.cuda.set_rng_state_all(state['cuda_rng_state_all'])
            resumed = True
            if verbose:
                print(
                    f'resuming {run_name} at epoch {start_epoch}; '
                    f'best epoch={best_epoch}, val_AP={best_ap:.4f}')
        elif verbose:
            print(f'ignored incompatible resume checkpoint: {resume_path.name}')

    epoch_range = range(start_epoch, max_epochs + 1) if stale < patience else ()
    for epoch in epoch_range:
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                logits = model(xb)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.detach()) * len(yb)

        val_scores, val_labels = predict_planet_probs(model, val_loader, device)
        val_ap = float(average_precision_score(val_labels, val_scores))
        history.append({
            'epoch': epoch,
            'train_loss': running_loss / len(train_idx),
            'val_PR_AUC': val_ap,
        })
        improved = best_state is None or val_ap > best_ap + 1e-6
        if improved:
            best_ap, best_epoch, stale = val_ap, epoch, 0
            best_state = cpu_state_dict(model)
        else:
            stale += 1

        periodic_log = (
            verbose or
            (log_every is not None and log_every > 0 and
             (epoch == start_epoch or epoch % log_every == 0 or
              epoch == max_epochs or stale >= patience)))
        if periodic_log:
            print(
                f'{run_name} epoch {epoch:03d} '
                f'loss={history[-1]["train_loss"]:.4f} '
                f'val_AP={val_ap:.4f} best={best_ap:.4f}'
                + (' *' if improved else ''))

        if resume_path is not None:
            atomic_torch_save({
                'complete': False,
                'signature': signature,
                'completed_epoch': epoch,
                'model_state': cpu_state_dict(model),
                'optimizer_state': optimizer.state_dict(),
                'scaler_state': scaler.state_dict(),
                'best_model_state': best_state,
                'best_val_PR_AUC': float(best_ap),
                'best_epoch': int(best_epoch),
                'stale_epochs': int(stale),
                'history': history,
                'torch_rng_state': torch.get_rng_state(),
                'cuda_rng_state_all': (
                    torch.cuda.get_rng_state_all() if use_amp else None),
            }, resume_path)

        if trial is not None:
            trial.report(val_ap, step=epoch)
            if trial.should_prune():
                import optuna
                raise optuna.TrialPruned()
        if stale >= patience:
            if verbose:
                print(f'early stop; best epoch was {best_epoch}')
            break

    if best_state is None:
        raise RuntimeError(f'{run_name} has no best model state to finalize')
    model.load_state_dict(best_state)
    if final_path is not None:
        atomic_torch_save({
            'complete': True,
            'signature': signature,
            'run_name': run_name,
            'model_name': CONFIG['model'],
            'model_params': model_params,
            'training_params': training_params,
            'model_state': best_state,
            'bands': CONFIG['bands'],
            'lo': lo, 'hi': hi,
            'best_epoch': int(best_epoch),
            'best_val_PR_AUC': float(best_ap),
            'history': history,
            'split': split_meta,
            'dataset_fingerprint': data['metadata']['preprocessing_fingerprint'],
        }, final_path)
        if verbose:
            print(
                f'saved completed {run_name}: best epoch={best_epoch}, '
                f'val_AP={best_ap:.4f}')

    return {
        'name': run_name, 'checkpoint': str(final_path) if final_path else None,
        'model_params': model_params,
        'training_params': training_params,
        'best_val_PR_AUC': float(best_ap),
        'resumed': resumed, 'reused': False,
    }


In [ ]:
base_training = {
    key: CONFIG['training'][key]
    for key in ('batch_size', 'learning_rate', 'weight_decay')
}
RUNS = {}
if not EVALUATE_ONLY:
    RUNS['baseline'] = train_cnn(
        'baseline', CONFIG['model_params'], base_training)
else:
    for run_name in ('baseline', 'hpo'):
        checkpoint_path = OUT['models'] / f'{run_name}.pt'
        if not checkpoint_path.exists():
            continue
        _, checkpoint = load_cnn(checkpoint_path)
        RUNS[run_name] = {
            'name': run_name, 'checkpoint': str(checkpoint_path),
            'model_params': checkpoint['model_params'],
            'training_params': checkpoint['training_params'],
            'best_val_PR_AUC': checkpoint['best_val_PR_AUC'],
            'resumed': False, 'reused': True,
        }
    if not RUNS:
        raise RuntimeError('EVALUATE_ONLY found no completed CNN checkpoints')
    print('EVALUATE_ONLY loaded:', list(RUNS))


## 5. Optional Optuna HPO

HPO uses only the train band for weight fitting and validation PR-AUC for pruning and selection. The winning settings are retrained on the full train band. Test is not loaded.


In [ ]:
def atomic_pickle_dump(value, path):
    path = Path(path)
    temporary = path.with_name(f'.{path.name}.part')
    with temporary.open('wb') as handle:
        pickle.dump(value, handle)
    os.replace(temporary, path)


def persist_hpo_state(study, _trial=None):
    trials_path = OUT['metrics'] / 'hpo_trials.csv'
    trials_temporary = trials_path.with_name(f'.{trials_path.name}.part')
    study.trials_dataframe().to_csv(trials_temporary, index=False)
    os.replace(trials_temporary, trials_path)
    complete = [trial for trial in study.trials if trial.state.name == 'COMPLETE']
    if complete:
        best = study.best_trial
        atomic_json_dump({
            'best_trial': best.number,
            'best_value': best.value,
            'best_params': best.params,
        }, OUT['metrics'] / 'hpo_best_params.json')
    atomic_pickle_dump(study.sampler, OUT['root'] / 'optuna_sampler.pkl')


study = None
if CONFIG['hpo']['enabled'] and not EVALUATE_ONLY:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    # Every CNN block ends with MaxPool2d(2). A P-pixel patch therefore
    # permits at most floor(log2(P)) blocks: 32 -> 16 -> 8 -> 4 -> 2 -> 1.
    patch_pixels = int(X.shape[-1])
    max_valid_depth = int(np.floor(np.log2(patch_pixels)))
    if max_valid_depth < 2:
        raise ValueError(f'Patch size {patch_pixels} is too small for this CNN')
    print(f'valid CNN depth range for {patch_pixels}x{patch_pixels} patches: 2..{max_valid_depth}')

    def objective(trial):
        model_params = {
            'width': trial.suggest_categorical('width', [8, 16, 32]),
            'depth': trial.suggest_int('depth', 2, max_valid_depth),
            'dropout': trial.suggest_float('dropout', 0.1, 0.5),
        }
        if 2 ** model_params['depth'] > patch_pixels:
            raise optuna.TrialPruned(
                f"depth={model_params['depth']} is invalid for "
                f'{patch_pixels}x{patch_pixels} patches')
        if CONFIG['model'] == 'siamese_cnn':
            model_params.update({
                'head_dim': trial.suggest_categorical('head_dim', [64, 128, 256]),
                'head_dropout': trial.suggest_float('head_dropout', 0.1, 0.4),
            })
        training_params = {
            'batch_size': trial.suggest_categorical('batch_size', [64, 128, 256]),
            'learning_rate': trial.suggest_float(
                'learning_rate', 1e-4, 3e-3, log=True),
            'weight_decay': trial.suggest_float(
                'weight_decay', 1e-6, 1e-2, log=True),
        }
        run = train_cnn(
            f'hpo_trial_{trial.number}', model_params, training_params,
            max_epochs=CONFIG['hpo']['max_epochs'],
            patience=CONFIG['hpo']['patience'], trial=trial,
            train_fraction=CONFIG['hpo']['train_subsample'],
            save=False, verbose=False, log_every=5)
        print(
            f"trial {trial.number:03d} val_AP={run['best_val_PR_AUC']:.4f} "
            f'params={trial.params}')
        return run['best_val_PR_AUC']

    sampler_path = OUT['root'] / 'optuna_sampler.pkl'
    sampler = optuna.samplers.TPESampler(seed=CONFIG['seed'])
    if sampler_path.exists():
        try:
            with sampler_path.open('rb') as handle:
                sampler = pickle.load(handle)
            print('restored Optuna sampler state')
        except Exception as exc:
            warnings.warn(f'could not restore Optuna sampler: {exc}')

    study = optuna.create_study(
        direction='maximize', study_name=CONFIG['experiment_name'],
        storage=f"sqlite:///{OUT['root'] / 'optuna.db'}",
        load_if_exists=True, sampler=sampler,
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=5, n_warmup_steps=5))
    finished = sum(
        trial.state.name in ('COMPLETE', 'PRUNED')
        for trial in study.trials)
    remaining = max(int(CONFIG['hpo']['n_trials']) - finished, 0)
    if remaining:
        print(
            f'HPO target={CONFIG["hpo"]["n_trials"]}; '
            f'finished={finished}; running {remaining} additional trials')
        study.optimize(
            objective, n_trials=remaining,
            timeout=(CONFIG['hpo']['timeout_minutes'] or 0) * 60 or None,
            callbacks=[persist_hpo_state])
    else:
        print(
            f'HPO target already satisfied by {finished} '
            'completed/pruned trials; reusing study')
    persist_hpo_state(study)

    best = study.best_params
    model_keys = {'width', 'depth', 'dropout', 'head_dim', 'head_dropout'}
    best_model_params = {k: v for k, v in best.items() if k in model_keys}
    best_training_params = {k: v for k, v in best.items() if k not in model_keys}
    print(
        f'retraining/reusing HPO winner trial {study.best_trial.number} '
        'on the full training band')
    RUNS['hpo'] = train_cnn(
        'hpo', best_model_params, best_training_params)
elif EVALUATE_ONLY:
    print('EVALUATE_ONLY: Optuna and all fitting skipped')
else:
    print('HPO disabled; the baseline remains the only CNN run.')

print(
    'runs to evaluate:',
    {name: round(run['best_val_PR_AUC'], 4) for name, run in RUNS.items()})


## 6. XGBoost spatial stacking and validation selection

The CNN is scored on the separate stack band. XGBoost learns from those out-of-sample CNN scores and their within-band neighborhood statistics; validation is used for early stopping. The raw CNN and stacked variant compete on validation PR-AUC.


In [ ]:
from xgboost import XGBClassifier

def spatial_frame(role, scores):
    idx = split[role]
    return neighbour_features(
        split['xy'][idx], scores, ks=CONFIG['stacking']['ks'])

validation_rows = []
validation_predictions = pd.DataFrame({
    'system_index': data['system_index'][split['val']],
    'y': y[split['val']],
})

for run_name, run in RUNS.items():
    model, checkpoint = load_cnn(run['checkpoint'])
    stack_scores, stack_y = predict_planet_probs(
        model, predict_loaders['stack'], device)
    val_scores, val_y = predict_planet_probs(
        model, predict_loaders['val'], device)

    threshold, best_f1 = best_f1_threshold(val_y, val_scores)
    raw_metrics = binary_metrics(val_y, val_scores, threshold)
    validation_rows.append({
        'run': run_name, 'variant': 'cnn', 'threshold': threshold,
        'validation_best_F1': best_f1, **raw_metrics,
        'checkpoint': run['checkpoint'], 'stacker': None})
    validation_predictions[f'{run_name}__cnn'] = val_scores

    if CONFIG['stacking']['enabled']:
        X_stack = spatial_frame('stack', stack_scores)
        X_val = spatial_frame('val', val_scores)
        positives = int(stack_y.sum())
        negatives = len(stack_y) - positives
        scfg = CONFIG['stacking']
        stacker = XGBClassifier(
            n_estimators=scfg['n_estimators'], max_depth=scfg['max_depth'],
            learning_rate=scfg['learning_rate'], subsample=.9,
            colsample_bytree=.9, objective='binary:logistic',
            eval_metric='aucpr', tree_method='hist',
            scale_pos_weight=negatives / max(positives, 1),
            early_stopping_rounds=scfg['early_stopping_rounds'],
            random_state=CONFIG['seed'], n_jobs=-1)
        stacker.fit(X_stack, stack_y, eval_set=[(X_val, val_y)], verbose=False)
        val_xgb = stacker.predict_proba(X_val)[:, 1]
        stacker_path = OUT['models'] / f'{run_name}_xgb_spatial.joblib'
        temporary = stacker_path.with_suffix('.joblib.part')
        joblib.dump(stacker, temporary)
        os.replace(temporary, stacker_path)

        threshold, best_f1 = best_f1_threshold(val_y, val_xgb)
        xgb_metrics = binary_metrics(val_y, val_xgb, threshold)
        validation_rows.append({
            'run': run_name, 'variant': 'xgb_spatial',
            'threshold': threshold, 'validation_best_F1': best_f1,
            **xgb_metrics, 'checkpoint': run['checkpoint'],
            'stacker': str(stacker_path)})
        validation_predictions[f'{run_name}__xgb_spatial'] = val_xgb

VALIDATION = pd.DataFrame(validation_rows).sort_values(
    'PR_AUC', ascending=False).reset_index(drop=True)
validation_predictions.to_parquet(
    OUT['predictions'] / 'validation_predictions.parquet', index=False)
VALIDATION.to_csv(OUT['metrics'] / 'validation_candidates.csv', index=False)
display(VALIDATION[['run', 'variant', 'n', 'PR_AUC', 'ROC_AUC',
                    'F1', 'precision', 'recall', 'threshold']])

winner = VALIDATION.iloc[0].to_dict()
FINAL_SELECTION = {
    'run': winner['run'], 'variant': winner['variant'],
    'validation_PR_AUC': float(winner['PR_AUC']),
    'threshold': float(winner['threshold']),
    'checkpoint': winner['checkpoint'],
    'stacker': winner['stacker'],
    'selection_metric': 'validation PR_AUC',
    'split': split_meta,
    'selected_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
atomic_json_dump(FINAL_SELECTION, FINAL_SELECTION_PATH)
print('FROZEN SELECTION')
print(json.dumps(FINAL_SELECTION, indent=2))


### 6a. Validation comparison of every model variant

The comparison below uses each candidate's own validation-fitted threshold. `PR_AUC` and `ROC_AUC` are threshold-free; the remaining classification metrics use that frozen candidate-specific threshold. HPO rows require `CONFIG['hpo']['enabled'] = True`.


In [ ]:
variant_names = {
    ('baseline', 'cnn'): 'baseline',
    ('baseline', 'xgb_spatial'): 'xgb_spatial',
    ('hpo', 'cnn'): 'hpo',
    ('hpo', 'xgb_spatial'): 'hpo_xgb_spatial',
}
expected_variants = [
    'baseline', 'xgb_spatial', 'hpo', 'hpo_xgb_spatial'
]
comparison_columns = [
    'n', 'prevalence', 'threshold', 'PR_AUC', 'ROC_AUC',
    'precision', 'recall', 'F1', 'balanced_accuracy', 'specificity',
    'TN', 'FP', 'FN', 'TP',
]

comparison = VALIDATION.copy()
comparison['model_variant'] = [
    variant_names.get((run, variant), f'{run}_{variant}')
    for run, variant in zip(comparison['run'], comparison['variant'])
]
MODEL_COMPARISON = (
    comparison.set_index('model_variant')[comparison_columns]
    .reindex(expected_variants)
)
MODEL_COMPARISON.index.name = 'model_variant'
MODEL_COMPARISON.to_csv(
    OUT['metrics'] / 'validation_model_comparison.csv'
)

missing_variants = MODEL_COMPARISON.index[
    MODEL_COMPARISON['PR_AUC'].isna()
].tolist()
if missing_variants:
    print(
        'Unavailable variants:', missing_variants,
        "— enable CONFIG['hpo']['enabled'] and rerun for HPO comparisons."
    )

display(
    MODEL_COMPARISON.style
    .format({
        'n': '{:,.0f}', 'prevalence': '{:.4f}', 'threshold': '{:.4f}',
        'PR_AUC': '{:.4f}', 'ROC_AUC': '{:.4f}',
        'precision': '{:.4f}', 'recall': '{:.4f}', 'F1': '{:.4f}',
        'balanced_accuracy': '{:.4f}', 'specificity': '{:.4f}',
        'TN': '{:,.0f}', 'FP': '{:,.0f}', 'FN': '{:,.0f}', 'TP': '{:,.0f}',
    }, na_rep='—')
    .background_gradient(
        subset=['PR_AUC', 'ROC_AUC', 'F1', 'balanced_accuracy'],
        cmap='YlGn'
    )
)

rate_metrics = [
    'PR_AUC', 'ROC_AUC', 'precision', 'recall',
    'F1', 'balanced_accuracy', 'specificity',
]
available_comparison = MODEL_COMPARISON.dropna(subset=['PR_AUC'])
ax = available_comparison[rate_metrics].T.plot(
    kind='bar', figsize=(12, 6), width=.82
)
ax.set(
    ylabel='metric value', xlabel='', ylim=(0, 1.05),
    title='Planet validation comparison — all available model variants'
)
ax.grid(axis='y', alpha=.2)
ax.legend(title='model variant', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(
    OUT['figures'] / 'validation_model_comparison.png',
    dpi=180, bbox_inches='tight'
)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
val_y = y[split['val']]
for row in VALIDATION.itertuples():
    scores = validation_predictions[f'{row.run}__{row.variant}'].to_numpy()
    precision, recall, _ = precision_recall_curve(val_y, scores)
    ax.plot(recall, precision, label=f'{row.run}/{row.variant} AP={row.PR_AUC:.3f}')
ax.axhline(val_y.mean(), color='0.5', linestyle='--', label='prevalence')
ax.set(xlabel='recall', ylabel='precision', title='Validation only')
ax.legend(fontsize=8)
fig.savefig(OUT['figures'] / 'validation_pr_curves.png', dpi=180, bbox_inches='tight')
plt.show()


## 7. One-shot sealed test evaluation

Run this only after accepting the frozen selection above. No alternative model, stacker, or threshold is chosen from test results. If the completion marker already exists, start a genuinely new experiment rather than reopening this holdout for development.


In [ ]:
if FINAL_MARKER_PATH.exists():
    raise RuntimeError(
        f'Test was already opened for this experiment: {FINAL_MARKER_PATH}. '
        'Use the saved result or choose a new experiment_name.')

with FINAL_SELECTION_PATH.open() as handle:
    selection = json.load(handle)
model, checkpoint = load_cnn(selection['checkpoint'])
test_loader = make_loader(
    split['test'], CONFIG['training']['predict_batch_size'])
test_cnn, test_y = predict_planet_probs(model, test_loader, device)
if selection['variant'] == 'cnn':
    test_scores = test_cnn
elif selection['variant'] == 'xgb_spatial':
    stacker = joblib.load(selection['stacker'])
    X_test = spatial_frame('test', test_cnn)
    test_scores = stacker.predict_proba(X_test)[:, 1]
else:
    raise ValueError(f"Unknown frozen variant: {selection['variant']}")

threshold = float(selection['threshold'])
TEST_METRICS = binary_metrics(test_y, test_scores, threshold)
test_predictions = pd.DataFrame({
    'system_index': data['system_index'][split['test']],
    'y': test_y, 'cnn_score': test_cnn,
    'selected_score': test_scores,
    'prediction': (test_scores >= threshold).astype(np.int8),
})
test_predictions.to_parquet(
    OUT['predictions'] / 'final_test_predictions.parquet', index=False)
atomic_json_dump(TEST_METRICS, OUT['metrics'] / 'final_test_metrics.json')

predicted = (test_scores >= threshold).astype(np.int8)
cm = confusion_matrix(test_y, predicted, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['intact', 'damaged']).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f"Final test — {selection['run']} / {selection['variant']}")
fig.savefig(OUT['figures'] / 'final_test_confusion.png', dpi=180, bbox_inches='tight')
plt.show()

marker = {
    'selection': selection,
    'metrics': TEST_METRICS,
    'prediction_file': 'final_test_predictions.parquet',
    'completed_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
atomic_json_dump(marker, FINAL_MARKER_PATH)
display(pd.DataFrame([TEST_METRICS]))


## Done

The experiment folder now contains the fitted CNN checkpoint(s), optional Optuna study, XGBoost stacker(s), validation candidate table, frozen selection, and one-shot test outputs. Notebook 3 can consume `final_test_predictions.parquet` without reopening model selection.
